In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 250)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
target_registry = pd.DataFrame([
    {
        "target": "H1_violation",
        "heuristic": "H1",
        "canonical_name": "Visibility of system status",
        "positive_value": 1,
        "negative_value": 0
    },
    {
        "target": "H4_violation",
        "heuristic": "H4",
        "canonical_name": "Consistency and standards",
        "positive_value": 1,
        "negative_value": 0
    },
    {
        "target": "H5_violation",
        "heuristic": "H5",
        "canonical_name": "Error prevention",
        "positive_value": 1,
        "negative_value": 0
    },
    {
        "target": "H6_violation",
        "heuristic": "H6",
        "canonical_name": "Recognition rather than recall",
        "positive_value": 1,
        "negative_value": 0
    }
])

target_registry

,target,heuristic,canonical_name,positive_value,negative_value
0,H1_violation,H1,Visibility of system status,1,0
1,H4_violation,H4,Consistency and standards,1,0
2,H5_violation,H5,Error prevention,1,0
3,H6_violation,H6,Recognition rather than recall,1,0


In [3]:
label_levels = pd.DataFrame([
    {
        "label_level": "HUMAN_DIRECT",
        "description":
            "La fuente humana evaluó explícitamente la heurística Nielsen correspondiente.",
        "primary_training": True
    },
    {
        "label_level": "LLM_DIRECT",
        "description":
            "La fuente LLM evaluó explícitamente la heurística Nielsen correspondiente.",
        "primary_training": True
    },
    {
        "label_level": "HUMAN_PROXY",
        "description":
            "Evaluación humana relacionada conceptualmente, pero no equivalente a la heurística.",
        "primary_training": False
    },
    {
        "label_level": "EXPERIMENTAL_PROXY",
        "description":
            "Condición experimental relacionada con la heurística.",
        "primary_training": False
    },
    {
        "label_level": "RATING_PROXY",
        "description":
            "Rating de usabilidad relacionado con el constructo.",
        "primary_training": False
    },
    {
        "label_level": "REFERENCE_ONLY",
        "description":
            "La fuente aporta información metodológica o problemas, pero no Y canónica utilizable.",
        "primary_training": False
    },
    {
        "label_level": "PENDING",
        "description":
            "Falta acceso o información necesaria.",
        "primary_training": False
    }
])

label_levels

,label_level,description,primary_training
0,HUMAN_DIRECT,La fuente humana evaluó explícitamente la heurística Nielsen correspondiente.,True
1,LLM_DIRECT,La fuente LLM evaluó explícitamente la heurística Nielsen correspondiente.,True
2,HUMAN_PROXY,"Evaluación humana relacionada conceptualmente, pero no equivalente a la heurística.",False
3,EXPERIMENTAL_PROXY,Condición experimental relacionada con la heurística.,False
4,RATING_PROXY,Rating de usabilidad relacionado con el constructo.,False
5,REFERENCE_ONLY,"La fuente aporta información metodológica o problemas, pero no Y canónica utilizable.",False
6,PENDING,Falta acceso o información necesaria.,False


In [4]:
label_registry = pd.DataFrame([
    {
        "dataset": "AMUSED",
        "label_level": "PENDING",
        "label_source": "HUMAN_EXPERT",
        "unit": "interaction/task",
        "H1": "PROXY_CANDIDATE",
        "H4": None,
        "H5": "PROXY_CANDIDATE",
        "H6": "PROXY_CANDIDATE",
        "primary_training": False,
        "reason": "Acceso todavía pendiente."
    },

    {
        "dataset": "Catching UX Flaws",
        "label_level": "LLM_DIRECT",
        "label_source": "LLM_3_EVALUATIONS",
        "unit": "website",
        "H1": "DIRECT",
        "H4": "DIRECT",
        "H5": "DIRECT",
        "H6": "DIRECT",
        "primary_training": True,
        "reason":
            "Las cuatro heurísticas Nielsen aparecen explícitamente con IssueFound."
    },

    {
        "dataset": "29042675",
        "label_level": "PENDING",
        "label_source": "LLM/HUMAN_COMPARISON",
        "unit": "prototype",
        "H1": "DIRECT_CANDIDATE",
        "H4": "DIRECT_CANDIDATE",
        "H5": "DIRECT_CANDIDATE",
        "H6": "DIRECT_CANDIDATE",
        "primary_training": False,
        "reason":
            "Necesitamos separar de forma inequívoca qué bloque representa cada evaluador y qué valores pueden convertirse a Y."
    },

    {
        "dataset": "SERENE",
        "label_level": "REFERENCE_ONLY",
        "label_source": "HUMAN_EXPERT",
        "unit": "usability_issue",
        "H1": None,
        "H4": None,
        "H5": None,
        "H6": None,
        "primary_training": False,
        "reason":
            "Ground truth humano, pero el archivo disponible no asigna los 46 problemas explícitamente a H1/H4/H5/H6."
    },

    {
        "dataset": "P3M",
        "label_level": "LLM_DIRECT",
        "label_source": "GPT4",
        "unit": "screenshot",
        "H1": "DIRECT_POSITIVE",
        "H4": "DIRECT_POSITIVE",
        "H5": "DIRECT_POSITIVE",
        "H6": "DIRECT_POSITIVE",
        "primary_training": False,
        "reason":
            "Los defectos contienen Nielsen explícito; falta justificar que ausencia de defecto pueda interpretarse como etiqueta negativa."
    },

    {
        "dataset": "PHR",
        "label_level": "HUMAN_PROXY",
        "label_source": "4_EXPERTS",
        "unit": "portal/task",
        "H1": "REQUIRES_MAPPING",
        "H4": "REQUIRES_MAPPING",
        "H5": "REQUIRES_MAPPING",
        "H6": "REQUIRES_MAPPING",
        "primary_training": False,
        "reason":
            "H01-H08 son heurísticas adaptadas a PHR; H01 NO debe asumirse equivalente a Nielsen H1."
    },

    {
        "dataset": "Dashboards",
        "label_level": "EXPERIMENTAL_PROXY",
        "label_source": "CONTROLLED_MANIPULATION",
        "unit": "participant/dashboard/task",
        "H1": None,
        "H4": "PROXY_CANDIDATE",
        "H5": None,
        "H6": "PROXY_CANDIDATE",
        "primary_training": False,
        "reason":
            "Baseline/intervention e issue type representan condiciones experimentales, no Nielsen directo."
    },

    {
        "dataset": "UICrit + RICO",
        "label_level": "RATING_PROXY",
        "label_source": "HUMAN_RATINGS",
        "unit": "rico_id/task",
        "H1": None,
        "H4": "PROXY_CANDIDATE",
        "H5": None,
        "H6": "PROXY_CANDIDATE",
        "primary_training": False,
        "reason":
            "Learnability/design quality/usability son ratings, no violaciones Nielsen binarias."
    },

    {
        "dataset": "Zenodo 21891322",
        "label_level": "REFERENCE_ONLY",
        "label_source": "HEURISTIC_EVALUATION",
        "unit": "interface",
        "H1": "REFERENCE",
        "H4": "REFERENCE",
        "H5": "REFERENCE",
        "H6": "REFERENCE",
        "primary_training": False,
        "reason":
            "En los archivos descargados no encontramos una tabla Y reutilizable directamente."
    },

    {
        "dataset": "Cultural Heritage",
        "label_level": "PENDING",
        "label_source": None,
        "unit": None,
        "H1": None,
        "H4": None,
        "H5": None,
        "H6": None,
        "primary_training": False,
        "reason":
            "Solo se dispone actualmente del artículo, no del dataset estructurado."
    }
])

display(label_registry)

,dataset,label_level,label_source,unit,H1,H4,H5,H6,primary_training,reason
0,AMUSED,PENDING,HUMAN_EXPERT,interaction/task,PROXY_CANDIDATE,NaN,PROXY_CANDIDATE,PROXY_CANDIDATE,False,Acceso todavía pendiente.
1,Catching UX Flaws,LLM_DIRECT,LLM_3_EVALUATIONS,website,DIRECT,DIRECT,DIRECT,DIRECT,True,Las cuatro heurísticas Nielsen aparecen explícitamente con IssueFound.
2,29042675,PENDING,LLM/HUMAN_COMPARISON,prototype,DIRECT_CANDIDATE,DIRECT_CANDIDATE,DIRECT_CANDIDATE,DIRECT_CANDIDATE,False,Necesitamos separar de forma inequívoca qué bloque representa cada evaluador y qué valores pueden convertirse a Y.
3,SERENE,REFERENCE_ONLY,HUMAN_EXPERT,usability_issue,NaN,NaN,NaN,NaN,False,"Ground truth humano, pero el archivo disponible no asigna los 46 problemas explícitamente a H1/H4/H5/H6."
4,P3M,LLM_DIRECT,GPT4,screenshot,DIRECT_POSITIVE,DIRECT_POSITIVE,DIRECT_POSITIVE,DIRECT_POSITIVE,False,Los defectos contienen Nielsen explícito; falta justificar que ausencia de defecto pueda interpretarse como etiqueta negativa.
5,PHR,HUMAN_PROXY,4_EXPERTS,portal/task,REQUIRES_MAPPING,REQUIRES_MAPPING,REQUIRES_MAPPING,REQUIRES_MAPPING,False,H01-H08 son heurísticas adaptadas a PHR; H01 NO debe asumirse equivalente a Nielsen H1.
6,Dashboards,EXPERIMENTAL_PROXY,CONTROLLED_MANIPULATION,participant/dashboard/task,NaN,PROXY_CANDIDATE,NaN,PROXY_CANDIDATE,False,"Baseline/intervention e issue type representan condiciones experimentales, no Nielsen directo."
7,UICrit + RICO,RATING_PROXY,HUMAN_RATINGS,rico_id/task,NaN,PROXY_CANDIDATE,NaN,PROXY_CANDIDATE,False,"Learnability/design quality/usability son ratings, no violaciones Nielsen binarias."
8,Zenodo 21891322,REFERENCE_ONLY,HEURISTIC_EVALUATION,interface,REFERENCE,REFERENCE,REFERENCE,REFERENCE,False,En los archivos descargados no encontramos una tabla Y reutilizable directamente.
9,Cultural Heritage,PENDING,NaN,NaN,NaN,NaN,NaN,NaN,False,"Solo se dispone actualmente del artículo, no del dataset estructurado."


In [5]:
CATCHING_DIR = (
    PUBLIC_RAW
    / "04_catching_ux_flaws"
    / "results"
)

records = []

for path in sorted(
    CATCHING_DIR.rglob("*.json")
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    site = path.parent.name

    evaluation = (
        path.stem
        .lower()
        .replace(" ", "")
    )

    for item in data:

        records.append({
            "site": site,
            "evaluation": evaluation,
            "heuristic_raw": item.get("Heuristic"),
            "issue_found": item.get("IssueFound"),
            "severity": item.get("SeverityRating"),
            "source_file": str(
                path.relative_to(PUBLIC_RAW)
            )
        })

catching_raw = pd.DataFrame(records)

print(catching_raw.shape)

(895, 6)


In [6]:
def normalize_nielsen(value):

    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    text = re.sub(
        r"^\d+\.\s*",
        "",
        text
    )

    mapping = {
        "visibility of system status": "H1",
        "consistency and standards": "H4",
        "error prevention": "H5",
        "recognition rather than recall": "H6"
    }

    if text in mapping:
        return mapping[text]

    m = re.fullmatch(
        r"heuristic\s*(\d+)",
        text
    )

    if m:

        number = int(m.group(1))

        if number in {1, 4, 5, 6}:
            return f"H{number}"

    return None

In [7]:
catching_raw[
    "heuristic"
] = catching_raw[
    "heuristic_raw"
].apply(
    normalize_nielsen
)

catching_selected = catching_raw[
    catching_raw["heuristic"].notna()
].copy()

print(catching_selected.shape)

display(
    pd.crosstab(
        catching_selected["heuristic"],
        catching_selected["issue_found"]
    )
)

(358, 7)


issue_found,False,True
heuristic,,
H1,4,85
H4,19,70
H5,12,78
H6,35,55


In [8]:
catching_selected[
    "violation"
] = (
    catching_selected[
        "issue_found"
    ]
    .astype("boolean")
    .astype("Int8")
)

In [9]:
catching_labels_evaluator = (
    catching_selected[
        [
            "site",
            "evaluation",
            "heuristic",
            "violation",
            "severity"
        ]
    ]
    .copy()
)

print(
    catching_labels_evaluator.shape
)

(358, 5)


In [10]:
def majority_consensus(series):

    values = (
        pd.to_numeric(
            series,
            errors="coerce"
        )
        .dropna()
        .astype(int)
    )

    if len(values) == 0:
        return pd.NA

    positives = int(
        (values == 1).sum()
    )

    negatives = int(
        (values == 0).sum()
    )

    if positives > negatives:
        return 1

    if negatives > positives:
        return 0

    # Empate: no inventar etiqueta
    return pd.NA

In [11]:
catching_consensus_long = (
    catching_labels_evaluator
    .groupby(
        [
            "site",
            "heuristic"
        ]
    )
    .agg(
        n_votes=(
            "violation",
            "count"
        ),

        positive_votes=(
            "violation",
            "sum"
        ),

        consensus=(
            "violation",
            majority_consensus
        ),

        max_severity=(
            "severity",
            "max"
        )
    )
    .reset_index()
)

display(
    catching_consensus_long.head(30)
)

,site,heuristic,n_votes,positive_votes,consensus,max_severity
0,Business-Website,H1,3,3,1,3
1,Business-Website,H4,3,3,1,1
2,Business-Website,H5,3,3,1,2
3,Business-Website,H6,3,3,1,1
4,Discord-bot-website-template,H1,3,3,1,3
5,Discord-bot-website-template,H4,3,2,1,1
6,Discord-bot-website-template,H5,3,3,1,3
7,Discord-bot-website-template,H6,3,3,1,1
8,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1,3,0,0,0
9,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H4,3,0,0,0


In [12]:
pd.crosstab(
    catching_consensus_long[
        "heuristic"
    ],
    catching_consensus_long[
        "consensus"
    ],
    dropna=False
)

consensus,0,1,<NA>
heuristic,,,
H1,1,29,0
H4,2,27,1
H5,2,28,0
H6,11,19,0


In [13]:
catching_consensus_long[
    "n_votes"
].value_counts()

n_votes
3    118
2      2
Name: count, dtype: Int64

In [14]:
catching_targets = (
    catching_consensus_long
    .pivot(
        index="site",
        columns="heuristic",
        values="consensus"
    )
    .reset_index()
)

In [15]:
catching_targets = (
    catching_consensus_long
    .pivot(
        index="site",
        columns="heuristic",
        values="consensus"
    )
    .reset_index()
)

In [16]:
catching_targets = (
    catching_targets
    .rename(
        columns={
            "H1": "H1_violation",
            "H4": "H4_violation",
            "H5": "H5_violation",
            "H6": "H6_violation"
        }
    )
)

In [17]:
catching_targets[
    "source_dataset"
] = "catching_ux_flaws"

catching_targets[
    "label_source"
] = "LLM_CONSENSUS"

catching_targets[
    "label_level"
] = "LLM_DIRECT"

In [18]:
catching_targets = catching_targets[
    [
        "site",
        "source_dataset",
        "label_source",
        "label_level",
        "H1_violation",
        "H4_violation",
        "H5_violation",
        "H6_violation"
    ]
]

display(
    catching_targets
)

print(
    catching_targets.shape
)

heuristic,site,source_dataset,label_source,label_level,H1_violation,H4_violation,H5_violation,H6_violation
0,Business-Website,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1
1,Discord-bot-website-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1
2,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,0,0,0,0
3,academic-website-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1
4,academicpages.github.io,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1
5,airspace-bootstrap,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1
6,beginner-html-site-styled,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,<NA>,1,0
7,charity_responsive-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,0
8,conference-website-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,0
9,coreui-free-react-admin-template,catching_ux_flaws,LLM_CONSENSUS,LLM_DIRECT,1,1,1,1


(30, 8)


In [19]:
target_cols = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

for col in target_cols:

    print(
        "\n",
        col
    )

    print(
        catching_targets[
            col
        ].value_counts(
            dropna=False
        )
    )


 H1_violation
H1_violation
1    29
0     1
Name: count, dtype: Int64

 H4_violation
H4_violation
1       27
0        2
<NA>     1
Name: count, dtype: Int64

 H5_violation
H5_violation
1    28
0     2
Name: count, dtype: Int64

 H6_violation
H6_violation
1    19
0    11
Name: count, dtype: Int64


In [20]:
proxy_registry = pd.DataFrame([
    {
        "dataset": "Dashboards",
        "source_construct": "Order&Grouping",
        "candidate_target": "H4_violation",
        "mapping_type": "EXPERIMENTAL_PROXY",
        "accepted_for_primary_training": False
    },
    {
        "dataset": "Dashboards",
        "source_construct": "DataPresentation",
        "candidate_target": "H6_violation",
        "mapping_type": "EXPERIMENTAL_PROXY",
        "accepted_for_primary_training": False
    },
    {
        "dataset": "Dashboards",
        "source_construct": "DataUnderstanding",
        "candidate_target": "H6_violation",
        "mapping_type": "EXPERIMENTAL_PROXY",
        "accepted_for_primary_training": False
    },
    {
        "dataset": "UICrit + RICO",
        "source_construct": "learnability",
        "candidate_target": "H6_violation",
        "mapping_type": "RATING_PROXY",
        "accepted_for_primary_training": False
    },
    {
        "dataset": "UICrit + RICO",
        "source_construct": "design_quality_rating",
        "candidate_target": "H4_violation",
        "mapping_type": "RATING_PROXY",
        "accepted_for_primary_training": False
    }
])

proxy_registry

,dataset,source_construct,candidate_target,mapping_type,accepted_for_primary_training
0,Dashboards,Order&Grouping,H4_violation,EXPERIMENTAL_PROXY,False
1,Dashboards,DataPresentation,H6_violation,EXPERIMENTAL_PROXY,False
2,Dashboards,DataUnderstanding,H6_violation,EXPERIMENTAL_PROXY,False
3,UICrit + RICO,learnability,H6_violation,RATING_PROXY,False
4,UICrit + RICO,design_quality_rating,H4_violation,RATING_PROXY,False


In [21]:
phr_warning = """
PHR H01-H08 are dataset-specific adapted heuristics.

H01 must NOT automatically be interpreted as Nielsen H1.
H04 must NOT automatically be interpreted as Nielsen H4.

A semantic mapping requires the actual definitions/checklist
of each PHR heuristic.
"""

print(phr_warning)


PHR H01-H08 are dataset-specific adapted heuristics.

H01 must NOT automatically be interpreted as Nielsen H1.
H04 must NOT automatically be interpreted as Nielsen H4.

A semantic mapping requires the actual definitions/checklist
of each PHR heuristic.



In [22]:
target_registry.to_csv(
    REPORTS_DIR
    / "target_registry.csv",
    index=False
)

label_levels.to_csv(
    REPORTS_DIR
    / "label_evidence_levels.csv",
    index=False
)

label_registry.to_csv(
    REPORTS_DIR
    / "dataset_label_registry.csv",
    index=False
)

proxy_registry.to_csv(
    REPORTS_DIR
    / "proxy_label_registry.csv",
    index=False
)

catching_labels_evaluator.to_csv(
    INTERIM_DIR
    / "catching_labels_evaluator.csv",
    index=False
)

catching_consensus_long.to_csv(
    INTERIM_DIR
    / "catching_labels_consensus_long.csv",
    index=False
)

catching_targets.to_csv(
    INTERIM_DIR
    / "catching_targets.csv",
    index=False
)

In [23]:
files_expected = [
    REPORTS_DIR / "target_registry.csv",
    REPORTS_DIR / "label_evidence_levels.csv",
    REPORTS_DIR / "dataset_label_registry.csv",
    REPORTS_DIR / "proxy_label_registry.csv",

    INTERIM_DIR / "catching_labels_evaluator.csv",
    INTERIM_DIR / "catching_labels_consensus_long.csv",
    INTERIM_DIR / "catching_targets.csv"
]

for path in files_expected:

    print(
        path.name,
        "✅" if path.exists() else "❌"
    )

target_registry.csv ✅
label_evidence_levels.csv ✅
dataset_label_registry.csv ✅
proxy_label_registry.csv ✅
catching_labels_evaluator.csv ✅
catching_labels_consensus_long.csv ✅
catching_targets.csv ✅
